##### trunc()

- is primarily for **date columns** and has **limited format options**.
- This function `truncates a date or timestamp` to the `specified format (e.g. 'year', 'month', 'week', 'quarter')`.
  - `trunc(date, "month")`
  - `trunc(date, "year")`
  - `trunc(date, "week")`
  - `trunc(date, "quarter")`  
  - `Works only on Date type`
  - `Returns Date`
  - `Mostly used for monthly/yearly reporting`

##### Syntax

     trunc(date, format)

|  Argument    |   Description                 |
|--------------|-------------------------------|
|  date        | The **column** containing **date values**. |
|  format      | A literal string specifying the truncation unit. Valid options are **'year' (or 'yyyy', 'yy')**, **'quarter', 'month' (or 'mon', 'mm'), or 'week'**. |
|  Limitation  | It **cannot truncate** to the **day, hour, or minute**; using these formats will return **null**. |

**trunc() supports only:**
- 'year'
- 'yyyy'
- 'yy'
- 'month'
- 'mon'
- 'mm'
- It **does not** support **hour, minute, second**.

| Function       | Works With | Purpose                             |
| -------------- | ---------- | ----------------------------------- |
| `trunc()`      | Date       | truncate to **month/year/week/quarter**          |
| `date_trunc()` | Timestamp  | truncate to **year/month/day/hour/minute** |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import trunc, date_trunc, col, sum, current_date, current_timestamp, to_timestamp, to_date

##### 1) Truncate when Dataframe column, `DateType format`

**a) Truncating a Date / timestamp to Year, month, Quarter, week using `current_date()`**

In [0]:
df_trn = spark.range(1) \
    .select(current_date().alias('today'), \
            trunc(current_date(), "year").alias("current_year_01"), \
            trunc(current_date(), "YYYY").alias("current_year_02"), \
            trunc(current_date(), "YY").alias("current_year_03"), \
            trunc(current_date(), "month").alias("current_month_01"), \
            trunc(current_date(), "mon").alias("current_month_02"), \
            trunc(current_date(), "MM").alias("current_month_03"), \
            trunc(current_date(), "quarter").alias("current_month_quarter"), \
            trunc(current_date(), "week").alias("current_month_week"), \
            current_timestamp(),
            trunc(current_timestamp(), "Year").alias("Year"),
            trunc(current_timestamp(), "Month").alias("Month"),
            trunc(current_timestamp(), "quarter").alias("Quarter"),
            trunc(current_timestamp(), "Week").alias("Week"))

display(df_trn)

today,current_year_01,current_year_02,current_year_03,current_month_01,current_month_02,current_month_03,current_month_quarter,current_month_week,current_timestamp(),Year,Month,Quarter,Week
2026-03-11,2026-01-01,2026-01-01,2026-01-01,2026-03-01,2026-03-01,2026-03-01,2026-01-01,2026-03-09,2026-03-11T06:17:12.698Z,2026-01-01,2026-03-01,2026-01-01,2026-03-09


**b) Truncating a Date to Year, month, Quarter, week**

In [0]:
# create sample DataFrame
data = [("2020-03-15", 500),
        ("2021-08-26", 600),
        ("2021-08-21", 750),
        ("2022-05-19", 860),
        ("2023-11-28", 950),
        ("2024-09-30", 1050),
        ("2024-09-01", 1200),
        ("2024-09-11", 1300),
        ("2025-07-17", 1500)]

df_trunc = spark.createDataFrame(data, ["date_col", "amount"]) \
    .withColumn("date_col", col("date_col").cast("date"))

display(df_trunc)

date_col,amount
2020-03-15,500
2021-08-26,600
2021-08-21,750
2022-05-19,860
2023-11-28,950
2024-09-30,1050
2024-09-01,1200
2024-09-11,1300
2025-07-17,1500


In [0]:
trunc_day_yr = (df_trunc.select("date_col",
                                # Truncating a Date to Year
                                trunc("date_col", "year").alias("first_day_of_year"),
                                # Truncating a Date to Month
                                trunc("date_col", "month").alias("first_day_of_month"),
                                # Truncating a Date to Quarter
                                trunc("date_col", "quarter").alias("quarter_of_year"),
                                # Truncating a Date to Week
                                trunc("date_col", "week").alias("week_of_year"),
                                # trunc() does not support 'day' for TimestampType, returns null
                                trunc("date_col", "day").alias("trunc_of_day"),
                                date_trunc("day", col("date_col")).alias("date_trunc_ts_day")))

display(trunc_day_yr)

date_col,first_day_of_year,first_day_of_month,quarter_of_year,week_of_year,trunc_of_day,date_trunc_ts_day
2020-03-15,2020-01-01,2020-03-01,2020-01-01,2020-03-09,null,2020-03-15T00:00:00.000Z
2021-08-26,2021-01-01,2021-08-01,2021-07-01,2021-08-23,null,2021-08-26T00:00:00.000Z
2021-08-21,2021-01-01,2021-08-01,2021-07-01,2021-08-16,null,2021-08-21T00:00:00.000Z
2022-05-19,2022-01-01,2022-05-01,2022-04-01,2022-05-16,null,2022-05-19T00:00:00.000Z
2023-11-28,2023-01-01,2023-11-01,2023-10-01,2023-11-27,null,2023-11-28T00:00:00.000Z
2024-09-30,2024-01-01,2024-09-01,2024-07-01,2024-09-30,null,2024-09-30T00:00:00.000Z
2024-09-01,2024-01-01,2024-09-01,2024-07-01,2024-08-26,null,2024-09-01T00:00:00.000Z
2024-09-11,2024-01-01,2024-09-01,2024-07-01,2024-09-09,null,2024-09-11T00:00:00.000Z
2025-07-17,2025-01-01,2025-07-01,2025-07-01,2025-07-14,null,2025-07-17T00:00:00.000Z


- **month** returns `first day` of that `month`.
- **year** returns `January 1st` of that `year`.

In [0]:
df_trunc.groupBy(trunc("date_col", "month").alias("first_day_of_month")) \
        .agg(sum("amount").alias("total_sales")) \
        .display()

first_day_of_month,total_sales
2020-03-01,500
2021-08-01,1350
2022-05-01,860
2023-11-01,950
2024-09-01,3550
2025-07-01,1500


##### c) Truncating a Timestamp to Year, Month

In [0]:
# Create a DataFrame with a timestamp column
data = [("2022-07-25 12:45:55",),
        ("2023-01-01 08:56:25",),
        ("2022-12-31 23:59:59",),
        ("2023-02-01 09:35:48",),
        ("2023-03-01 21:49:58",)]

df_ts = spark.createDataFrame(data, ["timestamp"])
display(df_ts)

# Convert the timestamp column to a TimestampType
df_ts_type = df_ts.withColumn("timestamp", to_timestamp("timestamp"))
display(df_ts_type)

timestamp
2022-07-25 12:45:55
2023-01-01 08:56:25
2022-12-31 23:59:59
2023-02-01 09:35:48
2023-03-01 21:49:58


timestamp
2022-07-25T12:45:55.000Z
2023-01-01T08:56:25.000Z
2022-12-31T23:59:59.000Z
2023-02-01T09:35:48.000Z
2023-03-01T21:49:58.000Z


In [0]:
# Truncate the timestamp to month
df_ts_final = (df_ts_type \
    .withColumn("trunc_ts_year", trunc(col("timestamp"), "year"))
    .withColumn("trunc_ts_month", trunc(col("timestamp"), "month"))
    .withColumn("trunc_ts_quarter", trunc(col("timestamp"), "quarter"))
    # trunc() does not support 'day' for TimestampType, returns null
    .withColumn("trunc_ts_day", trunc(col("timestamp"), "day"))
    .withColumn("date_trunc_ts_day", date_trunc("day", col("timestamp")))
    # cannot truncate hours from a timestamp. trunc() does not support hour, minute, second.
    .withColumn("trunc_ts_hour", trunc(col("timestamp"), "hour"))
    .withColumn("hour_truncated", date_trunc("hour", "timestamp")))

# Display the result
display(df_ts_final)

timestamp,trunc_ts_year,trunc_ts_month,trunc_ts_quarter,trunc_ts_day,date_trunc_ts_day,trunc_ts_hour,hour_truncated
2022-07-25T12:45:55.000Z,2022-01-01,2022-07-01,2022-07-01,null,2022-07-25T00:00:00.000Z,null,2022-07-25T12:00:00.000Z
2023-01-01T08:56:25.000Z,2023-01-01,2023-01-01,2023-01-01,null,2023-01-01T00:00:00.000Z,null,2023-01-01T08:00:00.000Z
2022-12-31T23:59:59.000Z,2022-01-01,2022-12-01,2022-10-01,null,2022-12-31T00:00:00.000Z,null,2022-12-31T23:00:00.000Z
2023-02-01T09:35:48.000Z,2023-01-01,2023-02-01,2023-01-01,null,2023-02-01T00:00:00.000Z,null,2023-02-01T09:00:00.000Z
2023-03-01T21:49:58.000Z,2023-01-01,2023-03-01,2023-01-01,null,2023-03-01T00:00:00.000Z,null,2023-03-01T21:00:00.000Z


##### 2) Truncate when Dataframe column, not in DateType format

In [0]:
from pyspark.sql.functions import to_date, col

In [0]:
df_dt = spark.createDataFrame([("01-23-2019",),("06-24-2019",),("09-20-2019",),("12-31-2019",),("01-01-2020",)], ["dt_start"]) \
    .select(
        col("dt_start"),
        to_date(col("dt_start"), "MM-dd-yyyy").alias("toDate"),
        trunc(to_date(col("dt_start"), "MM-dd-yyyy"), "Year").alias("Year"),
        trunc(to_date(col("dt_start"), "MM-dd-yyyy"), "Month").alias("Month")
    )
df_dt.display()

dt_start,toDate,Year,Month
01-23-2019,2019-01-23,2019-01-01,2019-01-01
06-24-2019,2019-06-24,2019-01-01,2019-06-01
09-20-2019,2019-09-20,2019-01-01,2019-09-01
12-31-2019,2019-12-31,2019-01-01,2019-12-01
01-01-2020,2020-01-01,2020-01-01,2020-01-01
